# TP Jour 1 — Choix des Datasets Kaggle (Mono ou Multi-Sources), HDFS & Couche Bronze

Bienvenue dans ce fil rouge **Data Lakehouse** ! Vous êtes libres de choisir votre thématique métier et de combiner **un dataset multi-tables ou plusieurs datasets publics complémentaires sur Kaggle** afin de les croiser par jointure.

### Objectifs de la séance
1. **Cadrer votre sujet métier** et sélectionner **un ou plusieurs datasets Kaggle** (~50 à 500 Mo au total) partageant au moins un axe de jointure (date/heure, zone géographique, code ISO/catégorie, identifiant métier).
2. **Explorer le cluster HDFS réel** (`1 NameNode` + `3 DataNodes`) configuré avec `dfs.blocksize = 16 Mo` et `dfs.replication = 2`.
3. **Ingérer vos fichiers bruts multi-sources sur HDFS** (`/lakehouse/bronze/raw/<source>/...`) et observer physiquement leur découpage en blocs de 16 Mo.
4. **Réaliser un crash-test en direct** : couper `datanode2` et observer la détection de panne + l'auto-réparation vers le 3e DataNode en ~50 secondes.
5. **Construire votre couche Bronze** (`/lakehouse/bronze/`) au format **Delta Lake** avec traçabilité de la source (`_source_dataset`, `_source_file`, `_ingested_at`) et inspecter vos jobs sur **Spark Web UI** (`http://localhost:4040`).

---
### Fiche de cadrage de votre projet
- **Titre du sujet** — Marché des transferts entre championnats de football.
- **Dataset Kaggle** — `otaviofbrito/soccer-player-transfer-dataset`, cinq tables relationnelles : transferts, joueurs, clubs, ligues et affiliations par saison.
- **Clés de jointure** — Identifiant du joueur, identifiants des clubs de départ et d’arrivée, identifiant du championnat et saison.
- **Problématique** — Quels championnats recrutent entre eux et comment se répartissent les indemnités enregistrées ?
- **Variables** — Année du transfert, indemnité, type de transfert et championnat.

## 1. Téléchargement de vos Datasets Kaggle dans `/workspace/data_raw/<source>/`

Renseignez dans la liste `KAGGLE_DATASETS` un ou plusieurs identifiants Kaggle (ex. `["auteur1/dataset-principal", "auteur2/meteo-ou-referentiel"]`), ou déposez directement vos fichiers/sous-dossiers dans `data_raw/`.
- Chaque dataset est isolé dans son propre sous-dossier `data_raw/<nom-dataset>/` pour éviter toute collision de noms (`train.csv`, `metadata.csv`, etc.) entre plusieurs sources.
- **Critère important :** Vérifiez qu'au moins un fichier dépasse **20 à 32 Mo** afin d'observer sa division en plusieurs blocs HDFS de **16 Mo** !

In [1]:
import re
import shutil
from pathlib import Path

DATA_RAW = Path("/workspace/data_raw")
DATA_RAW.mkdir(parents=True, exist_ok=True)
SUPPORTED_EXTS = {".csv", ".parquet", ".json", ".jsonl", ".tsv"}

# Dataset Kaggle : transferts de joueurs de football
KAGGLE_DATASETS = [
    "otaviofbrito/soccer-player-transfer-dataset"
]

for slug in KAGGLE_DATASETS:
    slug = slug.strip()
    if not slug:
        continue
    subdir = re.sub(r"[^a-zA-Z0-9._-]+", "_", slug.split("/")[-1]).strip("_") or "dataset"
    target_dir = DATA_RAW / subdir
    existing = [f for f in target_dir.rglob("*") if f.is_file() and f.suffix.lower() in SUPPORTED_EXTS] if target_dir.exists() else []
    if not existing:
        import kagglehub
        print(f"Téléchargement de {slug} -> {target_dir}/ ...")
        cache_dir = Path(kagglehub.dataset_download(slug))
        target_dir.mkdir(parents=True, exist_ok=True)
        for src_file in sorted(cache_dir.rglob("*")):
            if src_file.is_file() and src_file.suffix.lower() in SUPPORTED_EXTS:
                dest_file = target_dir / src_file.relative_to(cache_dir)
                dest_file.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(src_file, dest_file)

local_files = sorted([f for f in DATA_RAW.rglob("*") if f.is_file() and f.suffix.lower() in SUPPORTED_EXTS])

assert len(local_files) > 0, (
    "Aucun fichier détecté dans /workspace/data_raw/. "
    "Renseignez `KAGGLE_DATASETS` ci-dessus ou utilisez `task data:download DATASET='auteur1/ds1,auteur2/ds2'`."
)

print(f"Inventaire local multi-sources dans {DATA_RAW} ({len(local_files)} fichier(s)) :")
for f in local_files:
    rel = f.relative_to(DATA_RAW)
    size_mb = f.stat().st_size / (1024 * 1024)
    expected_blocks = max(1, int(-(-f.stat().st_size // (16 * 1024 * 1024))))
    print(f" - {str(rel):<55} {size_mb:>7.2f} Mo  (~{expected_blocks} bloc(s) HDFS de 16 Mo)")

Téléchargement de otaviofbrito/soccer-player-transfer-dataset -> /workspace/data_raw/soccer-player-transfer-dataset/ ...


100%|██████████| 57.1M/57.1M [-00:50<00:00, -1.18MB/s]

Extracting files...


Inventaire local multi-sources dans /workspace/data_raw (5 fichier(s)) :
 - soccer-player-transfer-dataset/club_league.csv             2.91 Mo  (~1 bloc(s) HDFS de 16 Mo)
 - soccer-player-transfer-dataset/clubs.csv                   1.32 Mo  (~1 bloc(s) HDFS de 16 Mo)
 - soccer-player-transfer-dataset/leagues.csv                 0.06 Mo  (~1 bloc(s) HDFS de 16 Mo)
 - soccer-player-transfer-dataset/players.csv               125.33 Mo  (~8 bloc(s) HDFS de 16 Mo)
 - soccer-player-transfer-dataset/transfers.csv             114.77 Mo  (~8 bloc(s) HDFS de 16 Mo)


## 2. Démarrage de la `SparkSession` pré-configurée sur HDFS

Grâce au fichier `config/spark-defaults.conf`, Spark pointe automatiquement vers `hdfs://namenode:9000` avec **Delta Lake** activé.

### Exercice 2.1 — Initialiser `SparkSession` et contrôler la configuration HDFS
1. Créez une session Spark nommée `"J1_HDFS_Ingestion_Bronze"`.
2. Récupérez via `spark.conf.get(...)` la valeur de `"spark.hadoop.fs.defaultFS"`, `"spark.hadoop.dfs.blocksize"` et `"spark.hadoop.dfs.replication"`.

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Initialiser la session Spark
spark = (
    SparkSession.builder
    .appName("J1_HDFS_Ingestion_Bronze")
    .getOrCreate()
)

# Récupérer les paramètres HDFS
default_fs = spark.conf.get("spark.hadoop.fs.defaultFS")
block_size = spark.conf.get("spark.hadoop.dfs.blocksize")
replication = spark.conf.get("spark.hadoop.dfs.replication")

print("Spark version       :", spark.version)
print("Default FileSystem  :", default_fs)
print("HDFS Block Size (o) :", block_size)
print("HDFS Replication    :", replication)

:: loading settings :: url = jar:file:/opt/venv/lib/python3.15/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
io.delta#delta-spark_4.2_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-3dab233d-c830-478c-9401-b4f64ce46391;1.0
	confs: [default]
	found io.delta#delta-spark_4.2_2.13;4.4.1 in central
	found io.delta#delta-storage;4.4.1 in central
	found io.unitycatalog#unitycatalog-client;0.6.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.6.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta-kernel-api;4.4.1 in central
	found

Spark version       : 4.2.0
Default FileSystem  : hdfs://namenode:9000
HDFS Block Size (o) : 16777216
HDFS Replication    : 2


## 3. Copie des fichiers bruts vers HDFS & Inspection des Blocs de 16 Mo

La cellule suivante téléverse vos fichiers bruts de `/workspace/data_raw` vers `/lakehouse/bronze/raw/` sur HDFS, puis interroge le NameNode pour afficher la localisation exacte de chaque bloc de 16 Mo sur les 3 DataNodes (`datanode1`, `datanode2`, `datanode3`).

In [3]:
jvm = spark._jvm
hadoop_conf = spark._jsc.hadoopConfiguration()
fs = jvm.org.apache.hadoop.fs.FileSystem.get(hadoop_conf)
PathHDFS = jvm.org.apache.hadoop.fs.Path

raw_hdfs_dir = PathHDFS("/lakehouse/bronze/raw")
fs.mkdirs(raw_hdfs_dir)

for file_path in local_files:
    rel_posix = file_path.relative_to(DATA_RAW).as_posix()
    src = PathHDFS(f"file://{file_path}")
    dst = PathHDFS(f"/lakehouse/bronze/raw/{rel_posix}")
    fs.mkdirs(dst.getParent())
    fs.copyFromLocalFile(False, True, src, dst)
    print(f"Téléversé vers HDFS : {dst}")

Téléversé vers HDFS : /lakehouse/bronze/raw/soccer-player-transfer-dataset/club_league.csv
Téléversé vers HDFS : /lakehouse/bronze/raw/soccer-player-transfer-dataset/clubs.csv
Téléversé vers HDFS : /lakehouse/bronze/raw/soccer-player-transfer-dataset/leagues.csv
Téléversé vers HDFS : /lakehouse/bronze/raw/soccer-player-transfer-dataset/players.csv
Téléversé vers HDFS : /lakehouse/bronze/raw/soccer-player-transfer-dataset/transfers.csv


In [4]:
# Inspection récursive du découpage en blocs de 16 Mo sur l'ensemble des sources ingérées
remote_iter = fs.listFiles(raw_hdfs_dir, True)
statuses = []
while remote_iter.hasNext():
    statuses.append(remote_iter.next())
statuses.sort(key=lambda s: s.getLen(), reverse=True)
largest_hdfs_path = statuses[0].getPath().toString()

for status in statuses:
    path = status.getPath()
    rel_hdfs = path.toString().split("/lakehouse/bronze/raw/", 1)[-1]
    length_mb = status.getLen() / (1024 * 1024)
    blocks = fs.getFileBlockLocations(status, 0, status.getLen())
    print(f"Fichier : {rel_hdfs} ({length_mb:.2f} Mo | repl={status.getReplication()})")
    for idx, blk in enumerate(blocks):
        print(f"   Bloc #{idx} ({blk.getLength() / (1024*1024):5.2f} Mo) -> DataNodes : {list(blk.getHosts())}")

Fichier : soccer-player-transfer-dataset/players.csv (125.33 Mo | repl=2)
   Bloc #0 (16.00 Mo) -> DataNodes : ['datanode2', 'datanode3']
   Bloc #1 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode2']
   Bloc #2 (16.00 Mo) -> DataNodes : ['datanode3', 'datanode2']
   Bloc #3 (16.00 Mo) -> DataNodes : ['datanode3', 'datanode2']
   Bloc #4 (16.00 Mo) -> DataNodes : ['datanode3', 'datanode2']
   Bloc #5 (16.00 Mo) -> DataNodes : ['datanode2', 'datanode3']
   Bloc #6 (16.00 Mo) -> DataNodes : ['datanode2', 'datanode3']
   Bloc #7 (13.33 Mo) -> DataNodes : ['datanode2', 'datanode3']
Fichier : soccer-player-transfer-dataset/transfers.csv (114.77 Mo | repl=2)
   Bloc #0 (16.00 Mo) -> DataNodes : ['datanode3', 'datanode2']
   Bloc #1 (16.00 Mo) -> DataNodes : ['datanode3', 'datanode2']
   Bloc #2 (16.00 Mo) -> DataNodes : ['datanode3', 'datanode2']
   Bloc #3 (16.00 Mo) -> DataNodes : ['datanode2', 'datanode3']
   Bloc #4 (16.00 Mo) -> DataNodes : ['datanode2', 'datanode3']
   Bloc #5 (16.00 

### Questions d'analyse (à vérifier aussi sur http://localhost:9870)
1. Comparez sur l'interface Web du NameNode (`Utilities > Browse the file system`) le nombre de blocs de votre fichier le plus volumineux avec sa taille divisée par 16 Mo.
2. Un fichier de 3 Mo bloque-t-il réellement 16 Mo d'espace disque sur les DataNodes ? Pourquoi ?
3. Pourquoi avoir configuré `dfs.replication = 2` sur un cluster à `3` DataNodes ?

**Réponses**
1. Le fichier des joueurs pèse 125,33 Mio et occupe 8 blocs : sept blocs de 16 Mio et un dernier d’environ 13,33 Mio.
2. Un fichier de 3 Mio utilise environ 3 Mio par réplique, auxquels s’ajoutent les métadonnées.
3. Avec deux répliques sur trois nœuds, une panne simple laisse une copie disponible. Le troisième nœud permet de rétablir la réplication.

---
## 4. Crash-Test en Direct : Panne d'un DataNode et Auto-Réparation (~50s)

Dans `config/hdfs-site.xml`, le délai de détection d'un DataNode mort a été abaissé de 10m30s à **~50 secondes** :
$$2 \times \text{recheck-interval (10s)} + 10 \times \text{heartbeat.interval (3s)} = 50\text{ s}$$

### Protocole à réaliser depuis votre terminal :
1. Inspectez l'état initial des blocs : `task hdfs:fsck`
2. Arrêtez brutalement le DataNode 2 :
   ```bash
   task hdfs:crash-dn2   # équivalent à : docker stop datanode2
   ```
3. Observez l'onglet **Datanodes** sur `http://localhost:9870` pendant ~50s : dès que `datanode2` passe à **Dead**, le NameNode détecte les blocs sous-répliqués et ordonne leur copie automatique vers le 3e DataNode survivant !

### Exercice 4.1 — Vérifier la continuité de service et la *Data Locality*
Pendant que `datanode2` est éteint, lisez votre plus gros fichier (`largest_hdfs_path`) avec PySpark, affichez `df_check.count()` ainsi que `df_check.rdd.getNumPartitions()`, et expliquez le lien avec le nombre de blocs HDFS.

*(Pensez ensuite à rallumer le nœud avec `task hdfs:restore-dn2` !)*

In [6]:
print("Fichier cible du test de résilience :", largest_hdfs_path)

# Lire le plus gros fichier CSV depuis HDFS
df_check = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(largest_hdfs_path)
)

# Vérifier la continuité de service
print(f"Nombre total de lignes lues       : {df_check.count():,}")
print(f"Nombre de partitions Spark créées : {df_check.rdd.getNumPartitions()}")

Fichier cible du test de résilience : hdfs://namenode:9000/lakehouse/bronze/raw/soccer-player-transfer-dataset/players.csv


Nombre total de lignes lues       : 858,947
Nombre de partitions Spark créées : 12


J’ai lu **858 947 lignes** dans le fichier des joueurs. HDFS le stocke en **8 blocs**, alors que Spark a utilisé **12 partitions** pour le lire. Spark décide de ses partitions de lecture selon sa configuration et la taille des entrées. Pour conclure sur la panne, il reste à relever l’état de `datanode2` pendant le comptage et à vérifier les répliques avec `fsck`.

---
## 5. Construction de votre Couche Bronze en Delta Lake (`/lakehouse/bronze/`)

### Exercice 5.1 — Convertir vos fichiers bruts (mono ou multi-datasets) en tables Delta Lake Bronze
Dans une architecture **Médaillon multi-sources**, la couche **Bronze** conserve chaque table brute fidèle à sa source tout en l'enrichissant de métadonnées de lignage (*lineage*) indispensables avant les jointures du Jour 2 :
- `_ingested_at` : `F.current_timestamp()`
- `_source_dataset` : `F.lit(dataset_name)` *(identifie le dataset d'origine)*
- `_source_file` : `F.col("_metadata.file_path")`

> **Attention aux noms de colonnes Delta/Parquet :** Si vos fichiers CSV bruts contiennent des espaces ou caractères spéciaux dans les en-têtes, normalisez-les simplement en `snake_case` (ex. `.toDF(*[c.strip().lower().replace(" ", "_") for c in df.columns])`).

Complétez la cellule ci-dessous pour ingérer chacune des tables utiles de vos datasets dans `/lakehouse/bronze/<nom_table>`.

In [7]:
from pyspark.sql import functions as F
import re

# Les 5 fichiers football stockés dans HDFS
BRONZE_SOURCES = {
    "players": (
        "soccer-player-transfer-dataset",
        "/lakehouse/bronze/raw/soccer-player-transfer-dataset/players.csv"
    ),
    "transfers": (
        "soccer-player-transfer-dataset",
        "/lakehouse/bronze/raw/soccer-player-transfer-dataset/transfers.csv"
    ),
    "clubs": (
        "soccer-player-transfer-dataset",
        "/lakehouse/bronze/raw/soccer-player-transfer-dataset/clubs.csv"
    ),
    "leagues": (
        "soccer-player-transfer-dataset",
        "/lakehouse/bronze/raw/soccer-player-transfer-dataset/leagues.csv"
    ),
    "club_league": (
        "soccer-player-transfer-dataset",
        "/lakehouse/bronze/raw/soccer-player-transfer-dataset/club_league.csv"
    )
}

# Conversion CSV -> Delta Lake
for table_name, (dataset_name, raw_path) in BRONZE_SOURCES.items():

    bronze_delta_path = f"/lakehouse/bronze/{table_name}"

    # 1. Lire le fichier CSV
    df_raw = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(raw_path)
    )

    # 2. Ajouter la traçabilité du fichier d'origine
    df_raw = df_raw.withColumn(
        "_source_file",
        F.col("_metadata.file_path")
    )

    # 3. Normaliser les noms des colonnes
    new_columns = [
        re.sub(r"[^a-z0-9_]", "_", c.strip().lower())
        for c in df_raw.columns
    ]
    df_raw = df_raw.toDF(*new_columns)

    # 4. Ajouter les métadonnées d'ingestion
    df_raw = (
        df_raw
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_dataset", F.lit(dataset_name))
    )

    # 5. Enregistrer au format Delta Lake
    (
        df_raw.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(bronze_delta_path)
    )

    # 6. Vérifier le nombre de lignes
    cnt = spark.read.format("delta").load(bronze_delta_path).count()

    print(f"[BRONZE OK] {table_name} -> {cnt:,} lignes")

26/11/16 15:39:16 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

[BRONZE OK] players -> 858,947 lignes


[BRONZE OK] transfers -> 3,469,314 lignes


[BRONZE OK] clubs -> 12,963 lignes


[BRONZE OK] leagues -> 475 lignes


[BRONZE OK] club_league -> 127,980 lignes


In [8]:
# Auto-contrôle de fin de Jour 1 : au moins une table Delta valide dans /lakehouse/bronze/
bronze_root = PathHDFS("/lakehouse/bronze")
bronze_delta_tables = [
    s.getPath().getName()
    for s in fs.listStatus(bronze_root)
    if s.isDirectory() and fs.exists(PathHDFS(s.getPath(), "_delta_log"))
]
assert len(bronze_delta_tables) > 0, "Aucune table Delta détectée dans /lakehouse/bronze/ !"
print("Tables Delta Bronze validées :", bronze_delta_tables)

Tables Delta Bronze validées : ['club_league', 'clubs', 'leagues', 'players', 'transfers']
